# MP17 — Convolution Playground

Investigate kernels, padding, stride, dilation, feature maps and receptive fields. This expanded teaching experiment includes independent NumPy/PyTorch comparisons and a licensed aerial training crop. Learner mastery remains unassessed.

## Intuition and conventions
A kernel is a table of weights. Slide it over an image, multiply overlapping pixels by weights, and sum. Outputs form a feature map. Inputs here are single-channel HW float64 arrays, with zero padding and no bias. PyTorch uses NCHW; we add batch and channel axes for comparison.

Deep-learning libraries usually compute cross-correlation: weights are not reversed. Mathematical convolution reverses both kernel axes. This distinction disappears for symmetric mean kernels but matters for asymmetric derivative filters.

## Mathematics and hand example
For output indices i,j, y[i,j] = sum over a,b of K[a,b] X[i*s+a*d-p, j*s+b*d-p]. Here s is stride, d dilation and p padding. Values outside the input are zero. Dilation spaces taps; stride spaces output centers.

For input [[1,2,3],[4,5,6],[7,8,9]] and kernel [[1,0],[0,-1]], the first output is 1-5=-4. All four valid outputs are -4. Flipping the kernel gives +4.

For length n and kernel length k, output length is floor((n+2p-d(k-1)-1)/s)+1. Effective kernel span is d(k-1)+1. For n=7, k=3, p=1, s=2, d=2, the output is floor((7+2-4-1)/2)+1=3. Kernels larger than padded inputs are rejected.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from rssr.convolution_lab import correlate, output_size, receptive_field
x=np.arange(1,10).reshape(3,3)
y=correlate(x,[[1,0],[0,-1]])
np.testing.assert_array_equal(y,np.full((2,2),-4))
assert output_size(7,3,stride=2,padding=1,dilation=2)==3
print(y)

## Controlled impulse experiment
Run `python scripts/convolution_experiment.py` from the repository root. An impulse is zero everywhere except one unit pixel. The asymmetric kernel contains 1 through 9. Four configurations vary stride, padding and dilation. Predict their shapes before reading results. Integer-valued computations must match PyTorch exactly; this is stronger than visually similar images. The impulse panels share a 0–9 scale.

In [ ]:
record=json.loads((ROOT/'reports/mp17-convolution.json').read_text())
assert all(row['torch_max_error']==0 for row in record['impulses'])
for row in record['impulses']: print(row)
from IPython.display import Image, display
display(Image(filename=str(ROOT/'reports/mp17-convolution.png')))

## Receptive-field span and support
A receptive field identifies input locations influencing one output. Let r be the bounding span and j the distance between adjacent output centers in input coordinates. Start r=1,j=1. A layer updates r_new=r+(k-1)*d*j, then j_new=j*s. Use the old j in the first equation.

A 3×3 stride-2 layer gives r=3,j=2. A subsequent 3×3 dilation-2 layer gives r=3+2*2*2=11,j=2. The bounding box is 11×11, but autograd measures only 81 contributing pixels: dilation leaves gaps. Span is not support count. Padding affects alignment and boundary overlap, not this interior-span equation.

Positive linear kernels isolate geometric support. In a trained nonlinear network, zero weights or activation gates may suppress paths, so gradients need not expose every possible geometric dependency.

In [ ]:
assert receptive_field([(3,2,1),(3,1,2)])=={'span':11,'jump':2}
assert record['receptive_field']['measured_span']==[11,11]
assert record['receptive_field']['contributing_pixels']==81
print(record['receptive_field'])

## Licensed real-data experiment
The script reads a checksummed 64×64 red-band crop from NAIP training ROI_00006, acquired 2021-09-17. NAIP is public domain; retain USDA/OpenSR attribution. Stored DN/10000 is numerical scaling, not calibrated reflectance. No new download or test scene is used.

Identity preserves pixels exactly. A normalized 3×3 mean smooths local contrast. A horizontal derivative responds to horizontal changes, often vertical edges. Zero padding introduces artificial boundary changes, visible in the derivative panel. Its signed color scale differs from the intensity panels. Derivative MSE relative to the input measures numerical change, not reconstruction quality. No learned model or SR improvement is claimed.

## Practice — solve before reading
1. Compute output length for n=8,k=3,p=1,s=2,d=1.
2. Describe a mean filter on a constant positive image, including boundaries.
3. Derive the span of three 3×3 stride-1, dilation-1 layers.
4. Why can stride-2 sampling miss a one-pixel feature?
5. Why is the measured support smaller than 121 pixels?

## Worked solutions
1. floor((8+2-2-1)/2)+1=4.
2. Interior stays constant because weights sum to one. Zero-padded boundaries darken because some inputs are zero.
3. Span increases 1→3→5→7, with jump one: a 7×7 box.
4. Output centers advance by two pixels; some alignments miss the feature. Prefiltering reduces aliasing but does not retain every detail.
5. Dilation leaves unsampled rows/columns even after composing these layers. The span uses extreme coordinates; support counts only contributing pixels.

## Reflection
Exact reference agreement establishes implementation mechanics, not usefulness for a task. Keep shape and geometry checks when introducing learned filters in later lessons.

References: [authoritative brief](../stage_05_cnn_sr/MP17_convolution_playground/README.md) and [PyTorch Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html).